# Getting data into MAUD

In [ ]:
from pathlib import Path
# Optional: shows how a template .par was made (data imported into the calibrated instrument).
# Templates are already provided in <sample>/template/template.par.
SAMPLE = "fe21_bcc_hcpCoexist"   # or "fe20_bccFe", "fe18_hcpFe"
esg_files = sorted(list((Path(SAMPLE) / "data" / "diff").glob("*00???_2d.esg")))

In [ ]:
esg_files

In [ ]:

from MILK.parallel import parallel_map

def run_maud_import(inputs):
    """run_MAUD_ins is a wrapper function to esg loader to enable parallel import."""
    from pathlib import Path
    from MILK.bin.esg_loader import main as esg_loader_main
    import numpy as np

    NDET=1
    TARGET = Path(inputs[2]) / "maud_import" / f"run{inputs[0]:03d}"
    MAUD_INPUT_PAR = str(Path("calibration") / "calibrated.par")
    MAUD_OUTPUT_PAR = TARGET / "Initial.par"
    MAUD_DETECTORS = ["PerkinElmer"]
    PONI_FILES = [None]*NDET
    ESG_FILES = [str(inputs[1])]
    if np.all([Path(esg).is_file() for esg in ESG_FILES]):
        esg_loader_main(esg_files=ESG_FILES,
                poni_files=PONI_FILES,
                maud_detectors=MAUD_DETECTORS,
                maud_input_par=MAUD_INPUT_PAR,
                maud_output_par=MAUD_OUTPUT_PAR,
                output=TARGET,
                maud_run_import=True
                )

_ = parallel_map(func=run_maud_import,
                        poolsize=2,
                        disable_tqdm=False,
                        loop_arg=[(i, f, SAMPLE) for i, f in enumerate(esg_files)]
                        )